# NPS-25-003

Template: Getting_started.ipynb from hepdata_lib

From hepdata_lib:
The following instructions and examples should get you started to get your analysis into [HEPData](https://hepdata.net) using `hepdata_lib`. Please also refer to the [documentation](http://hepdata-lib.readthedocs.io/). While you can also run `hepdata_lib` on your local computer, you can use the [binder](https://mybinder.org/) or [SWAN](http://swan.cern.ch/) services in the browser. Mind that SWAN is only available for people with a CERN account.

Also useful reference: https://github.com/jalimena/HepData_EXO-23-016/tree/main
See "main" function in createHepData_all.py

## General setup

To make sure things are working and `hepdata_lib` is available, run the following command:

In [19]:
import hepdata_lib
import numpy as np
from hepdata_lib import Submission, Table, Variable
from __future__ import print_function
print("hepdata_lib version", hepdata_lib.__version__)

hepdata_lib version 0.20.0


## Adding a table/figure

In HEPData, figures and table will both be `Table` objects. 

The first column is the mass of phi_2, the second of phi_1, and the third is the median upper limit.

Let's create the table/figure. First, we need to give it a name, which is usually just the identifier in the paper, i.e. "Figure _". The table also needs a description, which is usually the caption. You also need to describe the location, i.e. where to find it in the publication:

In [20]:
def make2DLimitTable(tableName, isBDT, fileName, imageName):

    table = Table(tableName)
    if isBDT:
        table.description = "95% CL observed upper limit on the cross section, using the BDT-based event categorization, as a function of scalar masses."
    else: 
        table.description = "95% CL observed upper limit on the cross section, using the cut-based event categorization, as a function of scalar masses."
   
    table.location = "Results"
    table.keywords["observables"] = ["SIG"]
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b"
    ]
    #do I need phrases and "particles"?
    data = np.loadtxt(f"NPS25003_inputs/{fileName}", skiprows=0)

    # Column meaning
    y_vals = data[:, 0]   # FIRST column = y bin centers
    x_vals = data[:, 1]   # SECOND column = x bin centers
    z_vals = data[:, 2]   # bin content

    # Build bin edges from centers
    def make_edges(centers):
        centers = np.unique(centers.astype(float))
        edges = np.zeros(len(centers) + 1)
        edges[1:-1] = 0.5 * (centers[1:] + centers[:-1])
        edges[0] = centers[0] - (edges[1] - centers[0])
        edges[-1] = centers[-1] + (centers[-1] - edges[-2])
        return centers, edges

    y_centers, y_edges = make_edges(y_vals)
    x_centers, x_edges = make_edges(x_vals)

    # Independent variables
    phi2_mass = Variable(
        "phi_2 mass",
        is_independent=True,
        is_binned=True,
        units="GeV"
    )

    phi1_mass = Variable(
        "phi_1 mass",
        is_independent=True,
        is_binned=True,
        units="GeV"
    )

    # Map center -> edge tuple
    y_edges_map = {y: (y_edges[i], y_edges[i+1]) for i, y in enumerate(y_centers)}
    x_edges_map = {x: (x_edges[i], x_edges[i+1]) for i, x in enumerate(x_centers)}

    # Only include bins that exist in your data
    phi2_mass.values = [y_edges_map[y] for y in y_vals]
    phi1_mass.values = [x_edges_map[x] for x in x_vals]

    # Dependent variable
    median_limit = Variable(
        "Median limit",
        is_independent=False,
        is_binned=False,
        units="pb"
    )

    median_limit.values = [float(v) for y,x,v in data] 
    median_limit.add_qualifier("SQRT(S)", "13", "TeV")

    # Add to table
    table.add_variable(phi1_mass)
    table.add_variable(phi2_mass)
    table.add_variable(median_limit)

    table.add_image(f"NPS25003_inputs/{imageName}")
    table.add_additional_resource("Original data file", f"NPS25003_inputs/{fileName}", copy_file=True) #to-do: replace with file and image name
    print(table.name)
    return table

In [21]:
def make1DLimitTable(tableName, fileName, imageName):
    """
    Reads a 7-column .txt file and creates a single 1D limit Table.
    isBDT and isCascade are inferred from the fileName path:
      - isBDT    = True if 'bdt'   in path, False if 'cutbased' in path
      - isCascade = True if '4b2t' in path, False if '2b2t' in path
    """
    # Infer flags from file path
    isBDT = "bdt" in fileName.lower()
    isCascade = "4b2t" in fileName.lower()

    table = Table(tableName)
    
    if isBDT:
        if isCascade:
            table.description = "Observed and expected 95% CL upper limits for the cascade scenario on the branching fraction B(H → φ1φ2 → 2τ2b), using the BDT-based event categorization and fit to the di-τ mass, as a function of the mass hypotheses (mφ1 , mφ2 )."
        else:
            table.description = "Observed and expected 95% CL upper limits for the non-cascade scenario on the branching fraction B(H → φ1φ2 → 2τ2b), using the BDT-based event categorization and fit to the di-τ mass, as a function of the mass hypotheses (mφ1 , mφ2 )."
    else: 
        if isCascade:
            table.description = "Observed and expected 95% CL upper limits for the cascade scenario on the branching fraction B(H → φ1φ2 → 2τ2b), using the cut-based event categorization and fit to the di-τ mass, as a function of the mass hypotheses (mφ1 , mφ2 )."
        else:
            table.description = "Observed and expected 95% CL upper limits for the non-cascade scenario on the branching fraction B(H → φ1φ2 → 2τ2b), using the cut-based event categorization and fit to the di-τ mass, as a function of the mass hypotheses (mφ1 , mφ2 )."
    

    table.location = "Results"
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b"
    ]

    data = np.loadtxt(f"NPS25003_inputs/{fileName}", skiprows=1)

    x_vals  = data[:, 0]
    obs     = data[:, 1]
    exp_m2s = data[:, 2]
    exp_m1s = data[:, 3]
    exp_med = data[:, 4]
    exp_p1s = data[:, 5]
    exp_p2s = data[:, 6]

    def make_edges(centers):
        centers = np.unique(centers.astype(float))
        edges = np.zeros(len(centers) + 1)
        edges[1:-1] = 0.5 * (centers[1:] + centers[:-1])
        edges[0] = centers[0] - (edges[1] - centers[0])
        edges[-1] = centers[-1] + (centers[-1] - edges[-2])
        return centers, edges

    x_centers, x_edges = make_edges(x_vals)
    x_edges_map = {x: (x_edges[i], x_edges[i + 1]) for i, x in enumerate(x_centers)}

    phi1_mass = Variable("phi_1 mass", is_independent=True, is_binned=True, units="GeV")
    phi1_mass.values = [x_edges_map[x] for x in x_vals]

    var_observed = Variable("Observed limit", is_independent=False, is_binned=False, units="pb")
    var_observed.values = [float(v) for v in obs]
    var_observed.add_qualifier("SQRT(S)", "13", "TeV")

    var_exp_med = Variable("Expected limit (median)", is_independent=False, is_binned=False, units="pb")
    var_exp_med.values = [float(v) for v in exp_med]
    var_exp_med.add_qualifier("SQRT(S)", "13", "TeV")

    var_exp_m1s = Variable("Expected limit (-1 sigma)", is_independent=False, is_binned=False, units="pb")
    var_exp_m1s.values = [float(v) for v in exp_m1s]
    var_exp_m1s.add_qualifier("SQRT(S)", "13", "TeV")

    var_exp_p1s = Variable("Expected limit (+1 sigma)", is_independent=False, is_binned=False, units="pb")
    var_exp_p1s.values = [float(v) for v in exp_p1s]
    var_exp_p1s.add_qualifier("SQRT(S)", "13", "TeV")

    var_exp_m2s = Variable("Expected limit (-2 sigma)", is_independent=False, is_binned=False, units="pb")
    var_exp_m2s.values = [float(v) for v in exp_m2s]
    var_exp_m2s.add_qualifier("SQRT(S)", "13", "TeV")

    var_exp_p2s = Variable("Expected limit (+2 sigma)", is_independent=False, is_binned=False, units="pb")
    var_exp_p2s.values = [float(v) for v in exp_p2s]
    var_exp_p2s.add_qualifier("SQRT(S)", "13", "TeV")

    table.add_variable(phi1_mass)
    table.add_variable(var_observed)
    table.add_variable(var_exp_med)
    table.add_variable(var_exp_m1s)
    table.add_variable(var_exp_p1s)
    table.add_variable(var_exp_m2s)
    table.add_variable(var_exp_p2s)

    table.add_image(f"NPS25003_inputs/{imageName}")
    table.add_additional_resource(
        "Original data file",
        f"NPS25003_inputs/{fileName}",
        copy_file=True
    )

    print(table.name)
    return table

In [22]:
def make1DLimitTable_v13(tableName, m1m2_pairs, subfolder, decay, isBDT, isCascade, imageName):
    """
    Creates a single combined 1D limit Table for one (method × topology) combination.
    
    Files are per-m1:  higgsCombine_a1a2_{decay}_allchannels_allyears_m1_{m1}_limits.txt
    Each file has columns: mh, obs, exp_m2s, exp_m1s, exp_med, exp_p1s, exp_p2s
    where mh is m2. We extract the row matching m2 from the appropriate m1 file.

    Parameters
    ----------
    tableName   : str
    m1m2_pairs  : list of (m1, m2) tuples defining the x-axis points
    subfolder   : str  e.g. "cutbased_root" or "bdtbased_root"
    decay       : str  "4b2t" or "2b2t"
    isBDT       : bool
    isCascade   : bool
    imageName   : str  path to summary plot (relative to NPS25003_inputs/)
    """
    table = Table(tableName)
    if isCascade:
        table.description = "B(H -> phi_1 phi_2 -> 2 tau 4b) (%)"
    else:
        table.description = "B(H -> phi_1 phi_2 -> 2 tau 2b) (%)"
    table.location = "Results"
    table.keywords["reactions"] = [
        "H -> phi_1 phi_2 -> 2 tau 4 b",
        "H -> phi_1 phi_2 -> 2 tau 2 b"
    ]

    # Cache loaded files so we don't re-read the same m1 file multiple times
    file_cache = {}

    rows = []
    for m1, m2 in m1m2_pairs:
        if m1 not in file_cache:
            fpath = f"NPS25003_inputs/{subfolder}/higgsCombine_a1a2_{decay}_allchannels_allyears_m1_{m1}_limits.txt"
            file_cache[m1] = np.loadtxt(fpath, skiprows=1)
        data = file_cache[m1]
        if data.ndim == 1:
            data = data[np.newaxis, :]  # single-row file edge case
        # mh column (col 0) is m2 — find the matching row
        match = data[np.isclose(data[:, 0], m2)]
        if len(match) == 0:
            raise ValueError(f"m2={m2} not found in m1={m1} file for {decay}/{subfolder}")
        row = match[0]
        obs, exp_m2s, exp_m1s, exp_med, exp_p1s, exp_p2s = (
            row[1], row[2], row[3], row[4], row[5], row[6]
        )
        rows.append((m1, m2, obs, exp_m2s, exp_m1s, exp_med, exp_p1s, exp_p2s))

    # Independent variable: (m1, m2) mass pair as a string label
    mass_pair_var = Variable(
        "(phi_1 mass, phi_2 mass)",
        is_independent=True,
        is_binned=False,
        units="GeV"
    )
    mass_pair_var.values = [f"({m1}, {m2})" for m1, m2, *_ in rows]

    var_observed = Variable("Observed limit",             is_independent=False, is_binned=False, units="pb")
    var_exp_med  = Variable("Expected limit (median)",    is_independent=False, is_binned=False, units="pb")
    var_exp_m1s  = Variable("Expected limit (-1 sigma)",  is_independent=False, is_binned=False, units="pb")
    var_exp_p1s  = Variable("Expected limit (+1 sigma)",  is_independent=False, is_binned=False, units="pb")
    var_exp_m2s  = Variable("Expected limit (-2 sigma)",  is_independent=False, is_binned=False, units="pb")
    var_exp_p2s  = Variable("Expected limit (+2 sigma)",  is_independent=False, is_binned=False, units="pb")

    for v in (var_observed, var_exp_med, var_exp_m1s, var_exp_p1s, var_exp_m2s, var_exp_p2s):
        v.add_qualifier("SQRT(S)", "13", "TeV")

    var_observed.values = [float(r[2]) for r in rows]
    var_exp_m2s.values  = [float(r[3]) for r in rows]
    var_exp_m1s.values  = [float(r[4]) for r in rows]
    var_exp_med.values  = [float(r[5]) for r in rows]
    var_exp_p1s.values  = [float(r[6]) for r in rows]
    var_exp_p2s.values  = [float(r[7]) for r in rows]

    table.add_variable(mass_pair_var)
    table.add_variable(var_observed)
    table.add_variable(var_exp_med)
    table.add_variable(var_exp_m1s)
    table.add_variable(var_exp_p1s)
    table.add_variable(var_exp_m2s)
    table.add_variable(var_exp_p2s)

    table.add_image(f"NPS25003_inputs/NPS-25-003/{imageName}")

    print(table.name)
    return table

## Main Function

The `Submission` object represents the whole HEPData entry and thus carries the top-level meta data that is equally valid for all the tables and variables you may want to enter. The object is also used to create the physical submission files you will upload to the HEPData web interface.

When using `hepdata_lib` to make an entry, you always need to create a `Submission` object. 

In [23]:
def main():
    submission = Submission()
    submission.read_abstract("NPS25003_inputs/abstract.txt")

    #ADL
    submission.add_additional_resource("ADL file", "NPS25003_inputs/NPS25003.adl", copy_file=True)

    #Production cross section
    submission.add_link("Standard ggF and VBF cross-sections from Handbook of LHC Cross-sections", "http://arxiv.org/abs/arXiv:1610.07922") 

    #Pythia configurations - not necessary since standard

    #Signal model UFO Files
    submission.add_link("Signal Model UFO files", "https://gitlab.com/apapaefs/twosinglet")

    #Generator Process cards
    submission.add_link("Generator Process Cards", "https://github.com/cms-sw/genproductions/pull/2705") 

    #Cut flow tables

    #Data distributions of relevant ML input features - not necessary since provided in paper

    #Small set of input vectors & ML outputs
    submission.add_link("BDT Models", "https://github.com/Aaravind96/aabbttBDT/tree/preservation/BDTmodels")
    
    #Signal Efficiencies for simplified models' model points - not sure if necessary

    #Statistical model
    submission.add_link("Datacards", "https://gitlab.cern.ch/cms-analysis/nps/nps-25-003/datacards/-/tree/master/input?ref_type=heads") 
    
    #Final wiki
    submission.add_link("Wiki", "https://cms-results.web.cern.ch/cms-results/public-results/publications/NPS-25-003/") 

    ##################
    # 2D Limit plots #
    ##################
    version_folder = "v11_limits"
    plots = "NPS-25-003"

    table_configs_2D = [
        # (name,                               isBDT, subfolder,    channel,         imageName)
        ("Fig_009-d_2D_BDT_allchannels",       True,  "bdt_based",   "allchannels", f"{plots}/Figure_009-d.pdf"),
        ("Fig_009-c_2D_BDT_emu",               True,  "bdt_based",   "emu",         f"{plots}/Figure_009-c.pdf"),
        ("Fig_009-a_2D_BDT_mutau",             True,  "bdt_based",   "mutau",       f"{plots}/Figure_009-a.pdf"),
        ("Fig_009-b_2D_BDT_etau",              True,  "bdt_based",   "etau",        f"{plots}/Figure_009-b.pdf"),
        ("Fig_015-d_2D_cutbased_allchannels",  False, "cut_based",   "allchannels", f"{plots}/Figure_015-d.pdf"),
        ("Fig_015-c_2D_cutbased_emu",          False, "cut_based",   "emu",         f"{plots}/Figure_015-c.pdf"),
        ("Fig_015-a_2D_cutbased_mutau",        False, "cut_based",   "mutau",       f"{plots}/Figure_015-a.pdf"),
        ("Fig_015-b_2D_cutbased_etau",         False, "cut_based",   "etau",        f"{plots}/Figure_015-b.pdf"),
    ]

    for name, isBDT, subfolder, channel, imageName in table_configs_2D:
        prefix = "bdt_" if isBDT else ""
        submission.add_table(make2DLimitTable(
            name,
            isBDT,
            f"{version_folder}/{subfolder}/median_limits_{channel}.txt",
            imageName 
        ))
        
    ##################
    # 1D Limit plots #
    ##################

    table_configs_1D = [
        # (name,                           subfolder,        decay,   m1)
        ("BDT_4b2t_m1_15",                "bdtbased_root",  "4b2t",  15),
        ("BDT_4b2t_m1_20",                "bdtbased_root",  "4b2t",  20),
        ("BDT_4b2t_m1_30",                "bdtbased_root",  "4b2t",  30),
        ("BDT_2b2t_m1_15",                "bdtbased_root",  "2b2t",  15),
        ("BDT_2b2t_m1_20",                "bdtbased_root",  "2b2t",  20),
        ("BDT_2b2t_m1_30",                "bdtbased_root",  "2b2t",  30),
        ("BDT_2b2t_m1_40",                "bdtbased_root",  "2b2t",  40),
        ("BDT_2b2t_m1_50",                "bdtbased_root",  "2b2t",  50),
        ("cutbased_4b2t_m1_15",           "cutbased_root",  "4b2t",  15),
        ("cutbased_4b2t_m1_20",           "cutbased_root",  "4b2t",  20),
        ("cutbased_4b2t_m1_30",           "cutbased_root",  "4b2t",  30),
        ("cutbased_2b2t_m1_15",           "cutbased_root",  "2b2t",  15),
        ("cutbased_2b2t_m1_20",           "cutbased_root",  "2b2t",  20),
        ("cutbased_2b2t_m1_30",           "cutbased_root",  "2b2t",  30),
        ("cutbased_2b2t_m1_40",           "cutbased_root",  "2b2t",  40),
        ("cutbased_2b2t_m1_50",           "cutbased_root",  "2b2t",  50),
    ]

    '''for name, subfolder, decay, m1 in table_configs_1D:
        prefix = "bdt_" if "bdt" in subfolder else ""
        fileName = f"{subfolder}/higgsCombine_a1a2_{decay}_allchannels_allyears_m1_{m1}_limits.txt"
        imageName = f"{plots}/plotLimit_{prefix}allyears_a1a2_{decay}_m1_{m1}_allchannels.pdf" 
        submission.add_table(make1DLimitTable(name, fileName, imageName))'''
        

    #######################
    # 1D Limit plots v13  #
    #######################

    _noncascade_pairs = [
        (15,20),(15,30),(20,30),(20,40),(30,40),(30,50),(30,60),
        (40,50),(40,60),(40,70),(40,80),(50,60),(50,70),
    ]
    _cascade_pairs = [
        (15,30),(15,40),(15,50),(15,60),(15,70),(15,80),(15,90),(15,100),(15,110),
        (20,40),(20,50),(20,60),(20,70),(20,80),(20,90),(20,100),
        (30,60),(30,70),(30,80),(30,90),
    ]

    table_configs_1D_v13 = [
        # (tableName,              subfolder,       decay,  isBDT, isCascade, pairs,         plot_tag,   imageName)
        ("Fig_013_1D_cutbased_cascade",    "cutbased_root", "4b2t", False, True,  _cascade_pairs,    "cutbased", "Figure_013.pdf"),
        ("Fig_007_1D_bdtbased_cascade",    "bdtbased_root", "4b2t", True,  True,  _cascade_pairs,    "bdt_",     "Figure_007.pdf"),
        ("Fig_014_1D_cutbased_noncascade", "cutbased_root", "2b2t", False, False, _noncascade_pairs, "cutbased", "Figure_014.pdf"),
        ("Fig_008_1D_bdtbased_noncascade", "bdtbased_root", "2b2t", True,  False, _noncascade_pairs, "bdt_",     "Figure_008.pdf"),
    ]

    for tableName, subfolder, decay, isBDT, isCascade, pairs, plot_tag, imageName in table_configs_1D_v13:
        #imageName = f"{plots}/plotLimit_{plot_tag}allyears_a1a2_{decay}_allchannels.pdf"
        submission.add_table(
            make1DLimitTable_v13(tableName, pairs, subfolder, decay, isBDT, isCascade, imageName)
        )
        
    table = Table("Cutflow")
    table.description = (
        "Cutflow showing event yields after each selection step for two "
        "signal mass points, labelled by (m1, m2) in GeV."
    )
    table.location = "Auxiliary material"
    table.keywords["observables"] = ["N"]

    # Independent variable: cut names
    cuts = Variable("Selection step", is_independent=True, is_binned=False, units="")
    cuts.values = [
        "No cuts applied",
        "After event pre-selections",
        "SR1_1b",
        "SR2_1b",
        "SR3_1b",
        "SR4_1b",
        "SR1_2b",
        "SR2_2b",
    ]
    table.add_variable(cuts)

    # Signal point (60, 40)
    sig_60_40 = Variable("Yield", is_independent=False, is_binned=False, units="")
    sig_60_40.values = [
        326685418,
        2146.033512,
        593.1758018,
        490.4246293,
        338.9056403,
        188.6788008,
        242.2365516,
        173.7265271,
    ]
    sig_60_40.add_qualifier("mass point (m1, m2)", "(60, 40) GeV")
    sig_60_40.add_qualifier("SQRT(S)", 13, "TeV")
    table.add_variable(sig_60_40)

    # Signal point (80, 30)
    sig_80_30 = Variable("Yield", is_independent=False, is_binned=False, units="")
    sig_80_30.values = [
        2780469921,
        75.53763655,
        29.13513019,
        15.4912386,
        7.48410232,
        3.17092978,
        9.127757683,
        7.005560957,
    ]
    sig_80_30.add_qualifier("mass point (m1, m2)", "(80, 30) GeV")
    sig_80_30.add_qualifier("SQRT(S)", 13, "TeV")
    table.add_variable(sig_80_30)
    submission.add_table(table) 

    for t in submission.tables:
        table.keywords["cmenergies"] = [13000]
    outdir = "NPS25003_output"
    print("Tables:", [t.name for t in submission.tables])
    
    submission.create_files(outdir, remove_old=True)
    

In [24]:
if __name__ == "__main__":
    main()

Fig_009-d_2D_BDT_allchannels
Fig_009-c_2D_BDT_emu
Fig_009-a_2D_BDT_mutau
Fig_009-b_2D_BDT_etau
Fig_015-d_2D_cutbased_allchannels
Fig_015-c_2D_cutbased_emu
Fig_015-a_2D_cutbased_mutau
Fig_015-b_2D_cutbased_etau
Fig_013_1D_cutbased_cascade
Fig_007_1D_bdtbased_cascade
Fig_014_1D_cutbased_noncascade
Fig_008_1D_bdtbased_noncascade
Tables: ['Fig_009-d_2D_BDT_allchannels', 'Fig_009-c_2D_BDT_emu', 'Fig_009-a_2D_BDT_mutau', 'Fig_009-b_2D_BDT_etau', 'Fig_015-d_2D_cutbased_allchannels', 'Fig_015-c_2D_cutbased_emu', 'Fig_015-a_2D_cutbased_mutau', 'Fig_015-b_2D_cutbased_etau', 'Fig_013_1D_cutbased_cascade', 'Fig_007_1D_bdtbased_cascade', 'Fig_014_1D_cutbased_noncascade', 'Fig_008_1D_bdtbased_noncascade', 'Cutflow']


In [25]:
!cat NPS25003_output/submission.yaml

---
additional_resources:
- description: Created with hepdata_lib 0.20.0
  location: https://doi.org/10.5281/zenodo.1217998
- description: ADL file
  location: NPS25003.adl
- description: Standard ggF and VBF cross-sections from Handbook of LHC Cross-sections
  location: http://arxiv.org/abs/arXiv:1610.07922
- description: Signal Model UFO files
  location: https://gitlab.com/apapaefs/twosinglet
- description: Generator Process Cards
  location: https://github.com/cms-sw/genproductions/pull/2705
- description: BDT Models
  location: https://github.com/Aaravind96/aabbttBDT/tree/preservation/BDTmodels
- description: Datacards
  location: https://gitlab.cern.ch/cms-analysis/nps/nps-25-003/datacards/-/tree/master/input?ref_type=heads
comment: A search for Higgs boson decays to a pair of neutral scalars phi_1 and phi_2
  with unequal masses is performed in final states with b quarks and tau leptons.
  Depending on the masses of the neutral scalars, phi_2 can undergo a cascade decay
  to a p

In [26]:
!ls NPS25003_output

Figure_007.png			     fig_015-a_2d_cutbased_mutau.yaml
Figure_008.png			     fig_015-b_2d_cutbased_etau.yaml
Figure_009-a.png		     fig_015-c_2d_cutbased_emu.yaml
Figure_009-b.png		     fig_015-d_2d_cutbased_allchannels.yaml
Figure_009-c.png		     median_limits_allchannels.txt
Figure_009-d.png		     median_limits_emu.txt
Figure_013.png			     median_limits_etau.txt
Figure_014.png			     median_limits_mutau.txt
Figure_015-a.png		     submission.yaml
Figure_015-b.png		     thumb_Figure_007.png
Figure_015-c.png		     thumb_Figure_008.png
Figure_015-d.png		     thumb_Figure_009-a.png
NPS25003.adl			     thumb_Figure_009-b.png
cutflow.yaml			     thumb_Figure_009-c.png
fig_007_1d_bdtbased_cascade.yaml     thumb_Figure_009-d.png
fig_008_1d_bdtbased_noncascade.yaml  thumb_Figure_013.png
fig_009-a_2d_bdt_mutau.yaml	     thumb_Figure_014.png
fig_009-b_2d_bdt_etau.yaml	     thumb_Figure_015-a.png
fig_009-c_2d_bdt_emu.yaml	     thumb_Figure_015-b.png
fig_009-d_2d_bdt_allchannels.yaml    thumb_Fig

In [27]:
!ls submission.tar.gz

submission.tar.gz
